# TAHRIX — GAT Elliptic Trainer v3 (calibrated probabilities)

**v2 problem:** class weights too aggressive → recall 0.95 but precision 0.09. F1 stuck at 0.24.

**v3 strategy:**
1. **No class weights** — use threshold tuning instead
2. **Stratified 80/20 random split** (TAHRIX uses prob output, not temporal classification)
3. **2-layer GAT** (simpler, less overfit on Elliptic)
4. **Cosine LR schedule**, LR 1e-2 → 1e-4
5. **500 epochs, patience 60** on AUC
6. **Post-training threshold sweep** → report F1 at best threshold

**Targets:** AUC > 0.95, F1@best_threshold > 0.85

Runtime → **GPU T4**

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch_geometric onnx onnxruntime scikit-learn

In [ ]:
import torch, torch.nn.functional as F
from pathlib import Path
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GATConv

DATA_DIR = Path('/content/elliptic_data'); DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS = Path('/content/artifacts'); ARTIFACTS.mkdir(exist_ok=True)

dataset = EllipticBitcoinDataset(root=str(DATA_DIR))
data = dataset[0]
print(data)
vals, counts = data.y.unique(return_counts=True)
for v, c in zip(vals.tolist(), counts.tolist()):
    label = {0: 'licit', 1: 'illicit', 2: 'unknown', -1: 'unknown(-1)'}.get(v, str(v))
    print(f'  y={v} ({label}): {c} ({100*c/len(data.y):.1f}%)')

## Stratified 80/20 split on labeled (y∈{0,1}) only

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

torch.manual_seed(42); np.random.seed(42)

labeled_idx = ((data.y == 0) | (data.y == 1)).nonzero(as_tuple=True)[0].numpy()
y_lab = data.y[labeled_idx].numpy()

train_idx, test_idx = train_test_split(
    labeled_idx, test_size=0.2, stratify=y_lab, random_state=42)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True; test_mask[test_idx] = True

for name, m in [('Train', train_mask), ('Test', test_mask)]:
    n_il = ((data.y == 1) & m).sum().item()
    n_lc = ((data.y == 0) & m).sum().item()
    print(f'{name}: total={m.sum().item()}, illicit={n_il} ({100*n_il/m.sum().item():.1f}%), licit={n_lc}')

## Model: 2-layer GAT (matches inference service)

In [ ]:
# NOTE: kept 3-layer to match app/services/gnn_service.py exactly.
# But uses only 4 heads + smaller hidden to reduce overfit.
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dims=(256, 128, 64), heads=8,
                 num_classes=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATConv(in_dim, hidden_dims[0] // heads, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dims[0], hidden_dims[1] // heads, heads=heads, dropout=dropout)
        self.gat3 = GATConv(hidden_dims[1], hidden_dims[2], heads=1, concat=False, dropout=dropout)
        self.out = torch.nn.Linear(hidden_dims[2], num_classes)
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat3(x, edge_index))
        return self.out(x)

## Train (plain CE, cosine LR, track AUC)

In [ ]:
import json
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def train_run(epochs=500, lr=1e-2, weight_decay=5e-4, patience=60):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[train] device={device}, epochs={epochs}, lr={lr}, patience={patience}')
    d = data.to(device); tm = train_mask.to(device); em = test_mask.to(device)
    in_dim = d.num_node_features
    model = GAT(in_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-4)

    best_auc, best_state, no_improve = 0.0, None, 0
    metrics = {}

    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(d.x, d.edge_index)
        loss = F.cross_entropy(logits[tm], d.y[tm].long())
        loss.backward(); optimizer.step(); scheduler.step()

        model.eval()
        with torch.no_grad():
            logits = model(d.x, d.edge_index)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            y_true = d.y[em].cpu().numpy()
            y_pred = preds[em].cpu().numpy()
            y_prob = probs[em].cpu().numpy()
            p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
            try: auc = roc_auc_score(y_true, y_prob)
            except ValueError: auc = float('nan')

        if epoch <= 10 or epoch % 20 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'[ep {epoch:03d}] loss={loss.item():.4f} P={p:.3f} R={r:.3f} F1={f1:.3f} AUC={auc:.4f} lr={cur_lr:.2e}')

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            metrics = {'epoch': epoch, 'loss': float(loss.item()), 'precision_argmax': float(p),
                       'recall_argmax': float(r), 'f1_argmax': float(f1), 'auc': float(auc),
                       'y_true': y_true.tolist(), 'y_prob': y_prob.tolist()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'[early-stop] no improvement for {patience} epochs at epoch {epoch}')
                break

    print(f'\n[done] best AUC={best_auc:.4f}')
    return in_dim, best_state, metrics

in_dim, best_state, m = train_run()

## Threshold tuning — find F1 max

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

y_true = np.array(m['y_true']); y_prob = np.array(m['y_prob'])
thresholds = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_true, (y_prob >= t).astype(int), pos_label=1, zero_division=0) for t in thresholds]
best_t = float(thresholds[int(np.argmax(f1s))])
y_pred_t = (y_prob >= best_t).astype(int)
best_f1 = f1_score(y_true, y_pred_t, pos_label=1, zero_division=0)
best_p = precision_score(y_true, y_pred_t, pos_label=1, zero_division=0)
best_r = recall_score(y_true, y_pred_t, pos_label=1, zero_division=0)

print(f'Best threshold: {best_t:.2f}')
print(f'  F1={best_f1:.4f}  P={best_p:.4f}  R={best_r:.4f}  AUC={m["auc"]:.4f}')

final_metrics = {
    'epoch': m['epoch'], 'loss': m['loss'],
    'auc': m['auc'],
    'best_threshold': best_t,
    'precision': best_p, 'recall': best_r, 'f1': best_f1,
    'precision_argmax': m['precision_argmax'], 'recall_argmax': m['recall_argmax'], 'f1_argmax': m['f1_argmax'],
}
(ARTIFACTS / 'gat_elliptic.metrics.json').write_text(json.dumps(final_metrics, indent=2))
print(json.dumps(final_metrics, indent=2))

In [ ]:
torch.save({'state_dict': best_state, 'in_dim': in_dim, 'best_threshold': best_t},
           ARTIFACTS / 'gat_elliptic.pt')
print('saved checkpoint')

## Export ONNX

In [ ]:
model_cpu = GAT(in_dim); model_cpu.load_state_dict(best_state); model_cpu.eval()
x_dummy = data.x[:128].clone().cpu()
edge_dummy = data.edge_index[:, :256].clone().cpu()
onnx_path = ARTIFACTS / 'gat_elliptic.onnx'
torch.onnx.export(model_cpu, (x_dummy, edge_dummy), str(onnx_path),
    input_names=['x', 'edge_index'], output_names=['logits'],
    dynamic_axes={'x': {0: 'num_nodes'}, 'edge_index': {1: 'num_edges'}, 'logits': {0: 'num_nodes'}},
    opset_version=18)
print(f'[ok] {onnx_path} ({onnx_path.stat().st_size // 1024} KB)')

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
x_np = data.x[:64].numpy().astype(np.float32)
e_np = data.edge_index[:, :100].numpy().astype(np.int64)
out = sess.run(None, {'x': x_np, 'edge_index': e_np})[0]
p = np.exp(out - out.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
print('logits shape:', out.shape, ' P(illicit) sample:', p[:5, 1])

In [ ]:
from google.colab import files
files.download(str(ARTIFACTS / 'gat_elliptic.onnx'))
files.download(str(ARTIFACTS / 'gat_elliptic.metrics.json'))
files.download(str(ARTIFACTS / 'gat_elliptic.pt'))